# Lorentz Boost of a Gravitational-Wave Tensor

This notebook demonstrates how source-frame gravitational-wave polarizations are transformed into detector-frame polarizations when the source and detector have a relativistic relative velocity. The implementation follows He, Liu, and Cao (2023), arXiv:2302.07532.

## Goal

Starting from the source-frame polarizations $h_+(t)$ and $h_\times(t)$, we:

1. construct a three-dimensional transverse-traceless tensor;
2. rotate the waveform into the detector polarization basis;
3. apply the exact Lorentz transformation;
4. calculate the aberrated propagation direction;
5. project the transformed tensor onto the new polarization basis; and
6. obtain the detector-frame waveform $H'_+(t')$ and $H'_\times(t')$.

The calculation follows the pipeline

$$
(h_+,h_\times)\rightarrow(H_+,H_\times)\rightarrow H_{ij}\rightarrow H'_{ij}\rightarrow(H'_+,H'_\times).
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from source import (
    M_SUN,
    KPC,
    source_waveform_from_quadrupole,
    source_h_to_detector_Hprime,
)

plt.rcParams.update({"figure.figsize": (8, 4.5), "font.size": 11})

## 1. Input parameters

The example uses a circular binary with component masses $m_1=m_2=30M_\odot$. The angles $\iota$, $\Theta$, $\Phi$, and $\Psi$ specify the binary inclination, source sky location, and polarization orientation.

The boost vector is $\boldsymbol{\beta}=\mathbf{v}/c$ in the detector Cartesian basis $(X,Y,Z)$. It must satisfy $|\boldsymbol{\beta}|<1$.

In [ ]:
# Binary parameters
m1 = 30.0 * M_SUN
m2 = 30.0 * M_SUN
f_orb = 20.0  # orbital frequency [Hz]
omega_orb = 2.0 * np.pi * f_orb
luminosity_distance = 10.0 * KPC

# Geometrical parameters
iota = np.deg2rad(60.0)
Theta = np.deg2rad(100.0)
Phi = np.deg2rad(40.0)
Psi = np.deg2rad(25.0)

# Source velocity relative to the detector, divided by c
beta_XYZ = np.array([0.1, 0.0, 0.0])

print(f"|beta| = {np.linalg.norm(beta_XYZ):.3f}")

## 2. Source waveform from the quadrupole formula

For a circular binary in the $x$-$y$ plane, the second derivative of the mass quadrupole is used to construct the far-zone strain tensor:

$$
h_{ij}^{\mathrm{raw}}(t)=\frac{2G}{D_Lc^4}\,\ddot{Q}_{ij}(t).
$$

The tensor is projected onto the plus and cross polarization tensors associated with the propagation direction $(\iota,0)$:

$$
h_+=\frac{1}{2}h_{ij}e_+^{ij},\qquad h_\times=\frac{1}{2}h_{ij}e_\times^{ij}.
$$

This produces the two source-frame polarization time series used as the input to the Lorentz transformation.

In [ ]:
# Simulate three orbital periods with 20 samples per orbit.
T_orb = 2.0 * np.pi / omega_orb
dt = 0.05 * T_orb
t = np.arange(0.0, 3.0 * T_orb + 0.5 * dt, dt)

hplus, hcross, hij_source, rhat_source = source_waveform_from_quadrupole(
    t=t,
    m1_kg=m1,
    m2_kg=m2,
    omega_orb=omega_orb,
    luminosity_distance_m=luminosity_distance,
    iota=iota,
)

print("Number of samples:", t.size)
print("h_ij shape:", hij_source.shape)

In [ ]:
fig, ax = plt.subplots()
ax.plot(t, hplus, label=r"$h_+$")
ax.plot(t, hcross, label=r"$h_\times$")
ax.set_xlabel(r"Source-frame time $t$ [s]")
ax.set_ylabel("Strain")
ax.set_title("Source-frame gravitational-wave polarizations")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 3. Polarization tensors

For a propagation direction $\hat{\mathbf{r}}$, the spherical basis vectors $\hat{\boldsymbol{\theta}}$ and $\hat{\boldsymbol{\phi}}$ span the plane transverse to the wave. The polarization tensors are

$$
e^+_{ij}=\hat{\theta}_i\hat{\theta}_j-\hat{\phi}_i\hat{\phi}_j,
$$

$$
e^\times_{ij}=\hat{\theta}_i\hat{\phi}_j+\hat{\phi}_i\hat{\theta}_j.
$$

They combine the two polarizations into a three-dimensional transverse-traceless tensor:

$$
h_{ij}=h_+e^+_{ij}+h_\times e^\times_{ij}.
$$

## 4. Rotation into the detector polarization basis

The polarization angle $\Psi$ rotates the source waveform into the detector sky basis:

$$
H_+=h_+\cos(2\Psi)+h_\times\sin(2\Psi),
$$

$$
H_\times=h_\times\cos(2\Psi)-h_+\sin(2\Psi).
$$

The factor of two occurs because a gravitational wave has spin weight 2. The components $H_+$ and $H_\times$ are then used to construct $H_{ij}$ in the detector basis.

## 5. Exact Lorentz boost and aberration

The code applies Eq. (59) of He, Liu, and Cao to transform $H_{ij}$ into $H'_{ij}$. This equation contains the original tensor, its contractions with the boost velocity, the Lorentz factor $\gamma$, and the propagation direction. Unlike a simple component-wise coordinate transformation, it preserves the physical transverse-traceless gravitational-wave degrees of freedom.

The propagation direction is transformed separately using the exact aberration relation in Eq. (79). From the aberrated direction, the code obtains the transformed sky angles $\Theta'$ and $\Phi'$ and constructs the new polarization basis.

Finally, $H'_{ij}$ is projected onto that basis to obtain $H'_+$ and $H'_\times$.

In [ ]:
result = source_h_to_detector_Hprime(
    h_plus=hplus,
    h_cross=hcross,
    iota=iota,
    theta=Theta,
    phi=Phi,
    psi=Psi,
    beta_XYZ=beta_XYZ,
)

print(f"Doppler factor k = {result['k']:.12g}")
print(f"Theta' = {np.rad2deg(result['Theta_prime']):.8f} deg")
print(f"Phi'   = {np.rad2deg(result['Phi_prime']):.8f} deg")

## 6. Time transformation

For a constant boost, Eq. (121) gives

$$
t'=kt,
$$

where

$$
k=\frac{1}{\gamma(1-\boldsymbol{\beta}\cdot\hat{\mathbf{r}})}.
$$

This changes the time scale, and therefore the observed frequency, of the waveform.

In [ ]:
t_prime = result["k"] * t

fig, ax = plt.subplots()
ax.plot(t_prime, result["Hplus_prime"], label=r"$H'_+$")
ax.plot(t_prime, result["Hcross_prime"], label=r"$H'_\times$")
ax.set_xlabel(r"Detector-frame time $t'$ [s]")
ax.set_ylabel("Strain")
ax.set_title("Lorentz-transformed detector-frame polarizations")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 7. Before-and-after comparison

The following plot compares the detector-basis polarizations immediately before and after the boost. The horizontal axis uses the corresponding time coordinate for each frame.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=False)

axes[0].plot(t, result["Hplus"], label=r"$H_+$ before boost")
axes[0].plot(t_prime, result["Hplus_prime"], "--", label=r"$H'_+$ after boost")
axes[0].set_ylabel("Strain")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(t, result["Hcross"], label=r"$H_\times$ before boost")
axes[1].plot(t_prime, result["Hcross_prime"], "--", label=r"$H'_\times$ after boost")
axes[1].set_xlabel("Time in the corresponding frame [s]")
axes[1].set_ylabel("Strain")
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.suptitle("Detector-basis waveform before and after the Lorentz boost")
fig.tight_layout()
plt.show()

## 8. Numerical validation

A physical gravitational-wave tensor must be traceless and transverse to its propagation direction. Therefore, after the boost we test

$$
\mathrm{Tr}(H')=0,\qquad H'_{ij}\hat{r}'^{j}=0.
$$

Both residuals should be close to zero within floating-point precision.

In [ ]:
trace_residual = np.trace(result["Hij_prime"], axis1=1, axis2=2)
propagation_direction_prime = -result["Rhat_prime"]
transverse_residual = np.einsum(
    "tij,j->ti", result["Hij_prime"], propagation_direction_prime
)

max_trace_error = np.max(np.abs(trace_residual))
max_transverse_error = np.max(
    np.linalg.norm(transverse_residual, axis=1)
)

print(f"Maximum |Tr(H')|:       {max_trace_error:.3e}")
print(f"Maximum |H' . r_hat'|: {max_transverse_error:.3e}")

## Reference

X. He, X. Liu, and Z. Cao, *Lorentz transformation of three dimensional gravitational wave tensor*, arXiv:2302.07532 (2023).

https://arxiv.org/abs/2302.07532